<a href="https://colab.research.google.com/github/MedushaThiru/Internship---DecodeLabs/blob/main/Project3_SQL_DecodeLabs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv('Cleaned_Dataset_Project1.csv')
df['Date'] = pd.to_datetime(df['Date'])

conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, index=False, if_exists='replace')

1200

Basic SELECT + WHERE

In [16]:
query = """
SELECT OrderID, Product, Quantity, TotalPrice
FROM orders
WHERE Quantity >5
ORDER BY TotalPrice DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,OrderID,Product,Quantity,TotalPrice


GROUP BY + aggregation (revenue per product)

In [4]:
query = """
SELECT Product, COUNT(*) AS NumOrders, SUM(TotalPrice) AS TotalRevenue, AVG(TotalPrice) AS AvgOrderValue
FROM orders
GROUP BY Product
ORDER BY TotalRevenue DESC;
"""
pd.read_sql(query, conn)

,Product,NumOrders,TotalRevenue,AvgOrderValue
0,Chair,178,195620.11,1098.989382
1,Printer,181,195612.61,1080.732652
2,Laptop,173,192126.56,1110.558150
3,Tablet,179,186568.95,1042.284637
4,Monitor,163,175651.41,1077.616012
5,Desk,170,167459.93,985.058412
6,Phone,156,151722.39,972.579423


GROUP BY on another dimension (e.g. ReferralSource)

In [5]:
query = """
SELECT ReferralSource, COUNT(*) AS Orders, SUM(TotalPrice) AS Revenue
FROM orders
GROUP BY ReferralSource
ORDER BY Revenue DESC;
"""
pd.read_sql(query, conn)

,ReferralSource,Orders,Revenue
0,Instagram,259,275285.45
1,Email,250,261808.55
2,Google,241,250441.48
3,Facebook,228,250410.90
4,Referral,222,226815.58


Monthly trend with GROUP BY + date function

In [6]:
query = """
SELECT strftime('%Y-%m', Date) AS Month, SUM(TotalPrice) AS MonthlyRevenue, COUNT(*) AS Orders
FROM orders
GROUP BY Month
ORDER BY Month;
"""
pd.read_sql(query, conn)

,Month,MonthlyRevenue,Orders
0,2023-01,56685.75,47
1,2023-02,40117.66,37
2,2023-03,48609.37,43
3,2023-04,27751.71,31
4,2023-05,63836.84,49
5,2023-06,49500.19,45
6,2023-07,42820.66,44
7,2023-08,54352.14,51
8,2023-09,29526.67,29
9,2023-10,52607.85,47


HAVING clause (the "experiment with unique solutions" part the conclusion mentions)

In [7]:
query = """
SELECT Product, SUM(TotalPrice) AS Revenue
FROM orders
GROUP BY Product
HAVING SUM(TotalPrice) > 180000
ORDER BY Revenue DESC;
"""
pd.read_sql(query, conn)

,Product,Revenue
0,Chair,195620.11
1,Printer,195612.61
2,Laptop,192126.56
3,Tablet,186568.95


Percentage contribution (the other bonus idea from the conclusion)

In [8]:
query = """
SELECT Product,
       SUM(TotalPrice) AS Revenue,
       ROUND(100.0 * SUM(TotalPrice) / (SELECT SUM(TotalPrice) FROM orders), 2) AS PctOfTotal
FROM orders
GROUP BY Product
ORDER BY Revenue DESC;
"""
pd.read_sql(query, conn)

,Product,Revenue,PctOfTotal
0,Chair,195620.11,15.47
1,Printer,195612.61,15.47
2,Laptop,192126.56,15.19
3,Tablet,186568.95,14.75
4,Monitor,175651.41,13.89
5,Desk,167459.93,13.24
6,Phone,151722.39,12.00


Payment method breakdown

In [9]:
query = """
SELECT PaymentMethod, COUNT(*) AS Orders, AVG(TotalPrice) AS AvgOrder
FROM orders
GROUP BY PaymentMethod
ORDER BY Orders DESC;
"""
pd.read_sql(query, conn)

,PaymentMethod,Orders,AvgOrder
0,Online,258,1017.220698
1,Cash,246,1056.041829
2,Credit Card,234,1127.553974
3,Debit Card,232,1001.556810
4,Gift Card,230,1070.973565


***This project used SQL queries (via SQLite in Python) to extract business insights from the cleaned orders dataset of 1,200 records. Grouping by product revealed that Chair ($195,620.11) and Printer ($195,612.61) generate the highest total revenue, each contributing roughly 15.5% of overall sales, while Phone contributed the least (12.00%, $151,722.39) despite having the highest average order value among the four products with the largest order counts. Applying a HAVING clause isolated four products — Chair, Printer, Laptop, and Tablet — as the top revenue generators, each exceeding $180,000. Analysis by referral source showed Instagram drove the most revenue ($275,285.45) from 259 orders, narrowly ahead of Email ($261,808.55), suggesting that social and email channels are the strongest acquisition sources. Given this, the business should prioritize inventory and marketing investment toward Chair, Printer, Laptop, and Tablet, while continuing to leverage Instagram and Email as primary referral channels.***